# TC-LoRA on cell states

Baseline 1 of 3. Trains a base diffusion model, freezes it, trains a TC-LoRA
hypernetwork on top, samples from both.

**Paper:** Cho, Ohana, Jacobsen, Jothi, Chen, Mao, Can.
*TC-LoRA: Temporally Modulated Conditional LoRA for Adaptive Diffusion Control.*
NeurIPS 2025 SpaVLE workshop. [arXiv:2510.09561](https://arxiv.org/abs/2510.09561)

**The authors released no code** (checked 2026-09-28). `src/guidance/baselines/tc_lora.py`
is our reimplementation from Eq. 1 and Appendices A-B; its header lists every deviation.
Report this as a reimplementation, not as their result reproduced.

**Label path.** Runs 1 and 2 gave the frozen backbone the true class label *and* gave the
hypernetwork a text embedding of the class name. Those carry the same information, so the
adapter had nothing to add (`docs/findings.md`). By default the backbone now sees the
**null label** and the text, through the adapter, is the only route to the class. Set
`label_path = True` to reproduce the run-2 setting. The cross-attention notebook uses the
same switch, so the two arms compare directly.

Run top to bottom. On Colab set **Runtime > Change runtime type > GPU** first. If you have
`base.pt` from an earlier run, upload it to the session first and base training is skipped.

## How this notebook implements TC-LoRA

**Notation.** $x_k$ is the noisy sample at diffusion step $k$ ($K = 1000$; the network receives
$t = k/K$), $\varepsilon_\theta$ the frozen noise-prediction MLP, $c$ the condition,
$\varnothing$ the null label, $h(c)$ the frozen CLIP embedding of the condition's description.
$\omega$ are the parameters trained in this notebook.

**The paper's idea (Eq. 1).** An ordinary LoRA adds a fixed low-rank correction $BA$ beside a
frozen weight $W$. TC-LoRA never learns $B$ and $A$ directly. It learns one hypernetwork
$H_\omega$ that *writes* them from the diffusion timestep, the condition, and the identity of the
layer, so the correction is different at every denoising step and for every condition:

$$W'_\ell(k, c) = W_\ell + \frac{\alpha_{\mathrm L}}{r}\, B_\ell A_\ell,
\qquad [A_\ell,\ B_\ell] = H_\omega\big(\gamma(k),\ h(c),\ e_\ell\big)$$

with rank $r = 4$, $\alpha_{\mathrm L} = 1$, $\gamma$ a sinusoidal embedding of the step and
$e_\ell$ a learned embedding of the layer's depth and type (App. A).

**How it is used here.** In the frozen MLP, the input projection and the two linear layers of each of the four residual blocks are adapted, nine layers; the time MLP, the drug and line embeddings and the output projection are not, because the paper adapts only the trunk's projections, the weight is replaced
by $W'_\ell(k, c)$ at forward time. The head of $H_\omega$ that writes $B$ is zero-initialised
(App. A), so at the start $B_\ell A_\ell = 0$ and the adapted model *is* the base model; the
notebook asserts this before training. Only $H_\omega$ trains, with the ordinary noise-prediction
loss (App. B), while the backbone receives the null label so that the text is its only route to
the condition:

$$\mathcal L(\omega) = \mathbb E_{k,\,x_0,\,\varepsilon}\Big[\big\|\varepsilon -
\varepsilon_{\theta,\,W'(k,c)}(x_k, k, \varnothing)\big\|_2^2\Big]$$

Sampling is the same DDIM step as the base model, with $\varepsilon_{\theta,W'(k,c)}$ in place
of $\varepsilon_\theta$: $N$ denoiser evaluations plus $N$ hypernetwork calls.

| In the paper | In this notebook |
|---|---|
| Eq. 1, $W'_\ell = W_\ell + B_\ell A_\ell$ with generated factors | `AdaptedLinear` / `AdaptedConv2d` in `tc_lora.py`, attached by `TCLoRA(...)` in *Attach TC-LoRA* |
| App. A, one hypernetwork shared by all layers, inputs $(k, h(c), e_\ell)$ | `HyperNet` in `tc_lora.py`; $h(c)$ is `COND_EMB` from *Frozen condition encoder* |
| App. A, zero-initialised $B$ head, adapted model equals the base at step 0 | the `allclose` assertion in *Attach TC-LoRA* |
| App. B, ordinary diffusion loss, hypernetwork the only trainable part | `tc_train`; `run_epochs(tc, [p ... if p.requires_grad])` |
| not in the paper: fan-in scaling of $A$, $\alpha_{\mathrm L}/r$ output scaling, step $\times 1000$ before embedding | deviations 6 and 7 in the `tc_lora.py` header, with the failure each one fixes |
| not in the paper: null label, wrong-text pass, matched validation | `COND`, `tc_val_wrong_text`, `validate`: our evaluation protocol, identical in the cross-attention notebook |

**What the paper did that this notebook does not.** Their backbone is a diffusion transformer
and the adapters sit on its attention projections; ours is a MLP, so for a
convolution the correction acts on the centre tap of each $3\times3$ kernel. Their condition is
a depth map passed through the base model's autoencoder; ours is a text embedding. They train
251M hypernetwork parameters for three days on eight H100s; ours has 1.7M and trains for
10 epochs on one GPU. Two things the paper leaves unspecified had to be added for training to
converge at all: the fan-in scaling of $A$, and scaling the step by 1000 before its sinusoidal
embedding, without which the hypernetwork barely sees time. All seven deviations are listed
in the header of `src/guidance/baselines/tc_lora.py`.

In [ ]:
# Colab bootstrap. Run once.
import importlib.util, subprocess, sys, os

for pkg in ("transformers", "datasets"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

if importlib.util.find_spec("guidance") is None:
    if not os.path.exists("discrete-generative-models-proj-1"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/imperorrp/discrete-generative-models-proj-1.git"],
                       check=True)
    sys.path.insert(0, "discrete-generative-models-proj-1/src")

import torch
from guidance.baselines.tc_lora import TCLoRA

# Record which commit of the code this run used. Goes into the manifest and the
# saved outputs, so two executed notebooks can be diffed by commit.
def _git_sha():
    for d in ("discrete-generative-models-proj-1", "."):
        try:
            return subprocess.check_output(["git", "-C", d, "rev-parse", "--short", "HEAD"],
                                           text=True, stderr=subprocess.DEVNULL).strip()
        except Exception:
            pass
    return "unknown"

GIT_SHA = _git_sha()
print("code version:", GIT_SHA)
print("device:", "cuda" if torch.cuda.is_available()
      else "CPU  <-- Runtime > Change runtime type > GPU")

## Config

In [ ]:
import math, time, json
from dataclasses import dataclass, asdict, fields
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Cfg:
    seed: int = 0          # training randomness
    split_seed: int = 0    # shuffling only; the Tahoe split is fixed by the data files
    batch_size: int = 256
    base_epochs: int = 30
    lora_epochs: int = 10
    condition: str = 'pair'       # pair (drug + cell line) | drug; see guidance.data.cells
    dose: float = -1.0             # -1 keeps every dose; 0.05, 0.5 or 5 keeps that dose (uM) plus controls
    lr_base: float = 2e-4
    lr_hyper: float = 1e-4
    K: int = 1000                  # diffusion training steps
    sample_steps: int = 50         # DDIM steps
    rank: int = 4
    hyper_width: int = 256
    label_path: bool = False       # False: backbone sees the null label; text is the only condition
    reuse_base: bool = True        # load base.pt if present instead of retraining
    limit: int = 0                 # >0 truncates the training set, for smoke tests

cfg = Cfg()

# Any field can be overridden from the environment, e.g. CFG_BASE_EPOCHS=1 CFG_LIMIT=5000, so
# headless runs (nbconvert, a server, a different GPU) never need a cell edited.
for _f in fields(Cfg):
    _v = os.environ.get(f"CFG_{_f.name.upper()}")
    if _v is not None:
        _cur = getattr(cfg, _f.name)
        setattr(cfg, _f.name, tuple(float(t) for t in _v.split(",")) if isinstance(_cur, tuple)
                else type(_cur)(_v) if not isinstance(_cur, bool) else _v.lower() in ("1", "true", "yes"))
        print(f"cfg.{_f.name} <- {getattr(cfg, _f.name)!r} (from CFG_{_f.name.upper()})")
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
print(DEV, asdict(cfg))

## Data

Tahoe-100M subset: 25 cell lines × 100 drugs at three doses, plus DMSO controls; 991,899 cells.
Each cell is 2000 highly variable genes reduced to 50 principal components, with normalisation,
gene selection, scaling and PCA all fitted on the training split only
(`notebooks/tahoe/tahoe_colab_downloader_with_pca.ipynb`). The split holds out about 20% of
(cell line, drug) **pairs**: every validation cell comes from a combination never seen in
training, while its line and its drug each appear in other training pairs. So validation loss
here already measures generalisation to unseen combinations. `x` is the 50-vector scaled so
its overall variance is 1.

The condition is the (cell line, drug) pair by default, 2525 ids, which the backbone turns into
a drug embedding plus a cell-line embedding, so an unseen pair is still a seen drug and a seen
line. Its text is the drug's mechanism of action and targets plus the cell line's name and
organ, from the Tahoe metadata, e.g. "Binimetinib, mek inhibitor targeting MAP2K1, MAP2K2, in
HepG2/C3A liver cells", written before any result. `condition = "drug"` drops the line.

**Data source.** Upload `25lines_100drugs_pca_data_v2.zip` to the session and run
`!unzip -q 25lines_100drugs_pca_data_v2.zip`, or mount Drive with the sidebar button. The
loader searches the usual places; set `TAHOE_PCA_DIR` if it is somewhere else.

In [ ]:
from guidance.data.cells import make_loaders

train_loader, val_loader, meta = make_loaders(
    batch_size=cfg.batch_size, seed=cfg.split_seed, condition=cfg.condition,
    dose=None if cfg.dose < 0 else cfg.dose, limit=cfg.limit)

xb, cb = next(iter(train_loader))
print("x", tuple(xb.shape), f"rms {float(xb.pow(2).mean().sqrt()):.2f}", "| cond", tuple(cb.shape))
print(f"train cells {meta['n_train']:,} | val cells {meta['n_val']:,} | "
      f"conditions {meta['n_conditions']} ({meta['condition']}) | split: {meta['split']}")
print("example text:", meta["cond_text"][1])

## Frozen condition encoder

`h(c)` is a frozen CLIP text embedding of the condition description. Falls back to fixed
random vectors if CLIP will not load, so the notebook still runs — but a fallback run is
**not a result**, and the manifest records it.

In [ ]:
def build_cond_embeddings(texts, device):
    try:
        from transformers import CLIPTokenizer, CLIPTextModel
        name = "openai/clip-vit-base-patch32"
        tok = CLIPTokenizer.from_pretrained(name)
        enc = CLIPTextModel.from_pretrained(name).to(device).eval()
        for p in enc.parameters():
            p.requires_grad_(False)
        out = []
        with torch.no_grad():
            for i in range(0, len(texts), 64):
                b = tok(texts[i:i + 64], padding=True, truncation=True, return_tensors="pt")
                out.append(enc(**{k: v.to(device) for k, v in b.items()}).pooler_output)
        emb = torch.cat(out)
        return emb / emb.norm(dim=-1, keepdim=True), False
    except Exception as e:
        print(f"CLIP unavailable ({type(e).__name__}); using random fallback embeddings.")
        g = torch.Generator().manual_seed(0)
        emb = torch.randn(len(texts), 512, generator=g)
        return (emb / emb.norm(dim=-1, keepdim=True)).to(device), True

COND_EMB, IS_FALLBACK = build_cond_embeddings(meta["cond_text"], DEV)
COND_DIM = COND_EMB.shape[1]
print("condition embeddings:", tuple(COND_EMB.shape), "| fallback:", IS_FALLBACK)

## Backbone

In [ ]:
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10_000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([args.cos(), args.sin()], dim=-1)


# ---- The model of notebooks/tahoe/Tahoe_PCA_Diffusion_CFG.ipynb, a teammate's classifier-free-
# guidance base for these cells, kept verbatim: drug and cell line are embedded separately and
# summed with the time embedding into one vector that enters every residual block.
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        half = dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half) / max(half - 1, 1))
        self.register_buffer("freqs", freqs, persistent=False)

    def forward(self, t):
        args = t.float()[:, None] * self.freqs[None, :].to(t.device)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResidualBlock(nn.Module):
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim + cond_dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.act = nn.SiLU()

    def forward(self, x, cond):
        h = torch.cat([self.norm(x), cond], dim=-1)
        h = self.fc2(self.act(self.fc1(h)))
        return x + h


class DiffusionMLP(nn.Module):
    def __init__(self, data_dim, n_drugs, n_lines, time_dim=128, cond_dim=128, hidden_dim=256, n_blocks=4):
        super().__init__()
        self.data_dim = data_dim
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))
        self.drug_embed = nn.Embedding(n_drugs + 1, cond_dim)   # last row is the null/unconditional id
        self.line_embed = nn.Embedding(n_lines + 1, cond_dim)   # last row is the null/unconditional id
        self.input_proj = nn.Linear(data_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResidualBlock(hidden_dim, cond_dim) for _ in range(n_blocks)])
        self.output_proj = nn.Linear(hidden_dim, data_dim)

    def forward(self, x, t, drug_id, line_id):
        cond = self.time_mlp(self.time_embed(t)) + self.drug_embed(drug_id) + self.line_embed(line_id)
        h = self.input_proj(x)
        for block in self.blocks:
            h = block(h, cond)
        return self.output_proj(h)


# ---- Our wrapper. The notebooks pass one cond_id and t in [0, 1]. The loader's table turns the
# id into (drug_id, line_id); the extra last row is the null condition, both ids null, which is
# their joint dropout. t is scaled back to the integer step their time embedding was written for.
COND_TABLE = torch.cat([meta["cond_ids_table"],
                        torch.tensor([[meta["n_drugs"], meta["n_lines"]]])]).to(DEV)
assert int(COND_TABLE.min()) >= 0, "this backbone needs condition = 'pair' or 'drug'"


class CellDiffusionMLP(DiffusionMLP):
    def __init__(self):
        super().__init__(meta["shape"][0], n_drugs=meta["n_drugs"], n_lines=meta["n_lines"])
        self.null = meta["n_conditions"]

    def forward(self, x, t, cond_ids=None):
        if cond_ids is None:
            cond_ids = torch.full((x.shape[0],), self.null, device=x.device, dtype=torch.long)
        ids = COND_TABLE[cond_ids]
        return super().forward(x, t * cfg.K, ids[:, 0], ids[:, 1])


def make_backbone():
    return CellDiffusionMLP()

# Only the trunk is adapted by TC-LoRA: the input projection and the residual blocks' two
# linear layers each. Time MLP, the two embeddings and the output projection stay untouched.
SKIP_NAMES = ("output_proj", "time_mlp", "drug_embed", "line_embed")

torch.manual_seed(cfg.seed)      # model init and training randomness follow `seed`
net = make_backbone().to(DEV)
print(f"backbone params: {sum(p.numel() for p in net.parameters()):,}")

## Diffusion — cosine schedule, epsilon prediction, DDIM sampler

In [ ]:
def cosine_alpha_bar(K, s=0.008, device="cpu"):
    steps = torch.arange(K + 1, dtype=torch.float32, device=device) / K
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0])[1:].clamp(1e-5, 0.9999)

ABAR = cosine_alpha_bar(cfg.K, device=DEV)
NULL_ID = meta["n_conditions"]

# At the first DDIM step sqrt(abar) is about 0.003, so x0 = (x - sqrt(1-abar) eps)/sqrt(abar)
# divides by a tiny number and any error in eps is amplified ~300x. Without clamping the
# samples blow up and the variance ratio reads in the thousands. Images are normalised to
# [-1, 1]; for standardised cell data the clamp is the training range.
CLIP_X0 = 1.0 if len(meta["shape"]) == 3 else float(
    torch.cat([x for x, _ in train_loader]).abs().max())
print("x0 clamp:", round(CLIP_X0, 3))


def add_noise(x0, gen=None):
    '''gen: pass a seeded torch.Generator to get the same (k, eps) every call.'''
    n = x0.shape[0]
    k = torch.randint(0, cfg.K, (n,), device=x0.device, generator=gen)
    eps = torch.randn(x0.shape, device=x0.device, generator=gen)
    ab = ABAR[k].view(n, *([1] * (x0.dim() - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, k, eps


def drop_labels(y, p_drop=0.1):
    '''Condition dropout creates the null branch that guidance needs later.'''
    keep = torch.rand(y.shape[0], device=y.device) >= p_drop
    return torch.where(keep, y, torch.full_like(y, NULL_ID))


@torch.no_grad()
def ddim_sample(eps_fn, n, shape, steps, clip_x0=None, n_snapshots=0, x_init=None):
    '''eps_fn(x, t) -> predicted noise. Deterministic DDIM.

    Convention, shared by every arm in this project: clip the predicted x0, then recompute
    eps from the clipped x0 before stepping (as TFG's official sampler does). Keeping the
    original eps is a different update whenever the clamp engages.

    x_init: pass the same starting noise to two calls for a paired comparison.
    n_snapshots > 0 also returns a list of (timestep, image) pairs for the trajectory
    figure: the starting noise, then the model's clean-image estimate x0_hat at that many
    evenly spaced steps. The last pair is the finished sample.
    '''
    clip_x0 = CLIP_X0 if clip_x0 is None else clip_x0
    idx = torch.linspace(cfg.K - 1, 0, steps).long().to(DEV)
    keep = set(torch.linspace(1, steps - 1, max(n_snapshots, 1)).long().tolist())
    x = torch.randn(n, *shape, device=DEV) if x_init is None else x_init.clone()
    snaps = [(cfg.K, x.cpu().clone())] if n_snapshots else []
    for i, k in enumerate(idx):
        t = (k.float() / cfg.K).repeat(n)
        eps = eps_fn(x, t)
        ab = ABAR[k]
        ab_prev = ABAR[idx[i + 1]] if i + 1 < len(idx) else torch.ones_like(ab)
        x0 = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-clip_x0, clip_x0)
        eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()          # recompute from the clipped x0
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
        if n_snapshots and i in keep:
            # x0_hat, not x_t: the noisy state looks like noise until the last few steps,
            # while the clean estimate shows what the model believes at every step.
            snaps.append((int(k), x0.cpu().clone()))
    return (x, snaps) if n_snapshots else x

## Base model — train, or load `base.pt` if it is already here

If `base.pt` from an earlier run is in the session, this cell loads it instead of
retraining, after checking it was trained under the same split, schedule and backbone. The
cross-attention notebook does the same, so both arms can share one frozen base.

Training and validation use **different** loss functions on purpose. Training applies 10%
label dropout so the null branch exists (and so the label-off setting is in-distribution for
the frozen model). Validation does not: it scores the model on the same fixed noise and
timesteps every epoch. The TC-LoRA arm is validated the same way. Without this, the base
model's validation number would include 10% harder unconditional predictions and the TC-LoRA
number would not, and the two would not be comparable. Run 1 had exactly that flaw.

In [ ]:
VAL_SEED = cfg.seed + 999


def run_epochs(model, params, n_epochs, lr, tag, train_fn, val_fn, ckpt):
    '''Trains with train_fn, validates with val_fn on fixed noise, keeps the best checkpoint.'''
    opt = torch.optim.AdamW(params, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(n_epochs, 1))
    hist, best, best_ep = [], float("inf"), 0
    for ep in range(1, n_epochs + 1):
        model.train(); tot = num = 0; t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEV), y.to(DEV)
            loss = train_fn(x, y)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            tot += loss.item() * x.shape[0]; num += x.shape[0]
        sched.step()

        val = validate(model, val_fn)
        is_best = val < best
        if is_best:
            best, best_ep = val, ep
            torch.save({"state": model.state_dict(), "epoch": ep, "val": val}, ckpt)
        hist.append({"epoch": ep, "train": tot / num, "val": val, "best": is_best,
                     "seconds": time.time() - t0})
        print(f"{tag} epoch {ep:3d}  train {tot/num:.4f}  val {val:.4f}"
              f"{'  *best' if is_best else ''}  ({time.time()-t0:.0f}s)")
    print(f"{tag}: best val {best:.4f} at epoch {best_ep}, saved to {ckpt}")
    return hist


@torch.no_grad()
def validate(model, val_fn):
    '''Same examples, same timesteps, same noise on every call -- for any model.'''
    model.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    tot = num = 0
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        tot += val_fn(x, y, gen).item() * x.shape[0]; num += x.shape[0]
    return tot / num


def base_train(x, y):
    xk, k, eps = add_noise(x)
    return F.mse_loss(net(xk, k.float() / cfg.K, drop_labels(y)), eps)


def base_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(net(xk, k.float() / cfg.K, y), eps)     # no dropout


# What a base.pt must agree on before it can be reused. A checkpoint trained under a
# different split would put its own training images into this run's validation set.
CKPT_META = {"split_seed": cfg.split_seed, "n_conditions": meta["n_conditions"],
             "shape": list(meta["shape"]), "K": cfg.K, "schedule": "cosine",
             "backbone_params": sum(p.numel() for p in net.parameters())}

import os
if cfg.reuse_base and os.path.exists("base.pt"):
    ck = torch.load("base.pt", map_location=DEV)
    saved = ck.get("meta")
    if saved is None:
        print("WARNING: base.pt carries no setup metadata (older run); split and schedule "
              "cannot be verified. Treat any comparison against it with care.")
    else:
        mismatch = {k: (saved.get(k), v) for k, v in CKPT_META.items() if saved.get(k) != v}
        assert not mismatch, f"base.pt was trained under a different setup: {mismatch}"
    net.load_state_dict(ck["state"])
    base_hist = [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val"], "best": True}]
    print(f"loaded base.pt (epoch {ck['epoch']}, val {ck['val']:.4f}); skipping base training")
else:
    base_hist = run_epochs(net, list(net.parameters()), cfg.base_epochs, cfg.lr_base,
                           "base", base_train, base_val, "base.pt")
    # Stamp the checkpoint with what it was trained under, so a later notebook can check it.
    ck = torch.load("base.pt", map_location=DEV); ck["meta"] = CKPT_META; torch.save(ck, "base.pt")

## Attach TC-LoRA and train the hypernetwork

The backbone is frozen; only the hypernetwork trains. The first assertion is the one that
matters — at initialisation the adapted model must match the base exactly, because the
hypernetwork's B head is zero-initialised (Appendix A). If that fails, the adapter is not
a clean control and the comparison means nothing.

`COND(y)` is the label the backbone sees: the true label if `cfg.label_path`, otherwise the
null token. With the null token, the adapter is the model's only knowledge of the class.

In [ ]:
ckpt = torch.load("base.pt", map_location=DEV)
print(f"loading base from epoch {ckpt['epoch']} (val {ckpt['val']:.4f})")
base_frozen = make_backbone().to(DEV)
base_frozen.load_state_dict(ckpt["state"])
base_frozen.eval()

def COND(y):
    '''The label the frozen backbone sees. Null by default, so the class can only reach the
    model through the adapter. cfg.label_path = True gives the backbone the true label.'''
    return y if cfg.label_path else torch.full_like(y, NULL_ID)

probe_x, probe_y = xb[:4].to(DEV), cb[:4].to(DEV)
probe_t = torch.rand(4, device=DEV)
with torch.no_grad():
    ref = base_frozen(probe_x, probe_t, COND(probe_y)).clone()

# Eq. 1: every trunk Linear / Conv2d (not the time MLP, label embedding or output) becomes
# W + B(t, h(c), l) A(t, h(c), l),
# with B and A written by one shared hypernetwork. The backbone's own weights stay frozen.
tc = TCLoRA(base_frozen, cond_dim=COND_DIM, rank=cfg.rank,
            width=cfg.hyper_width, skip_names=SKIP_NAMES).to(DEV)

# App. A: the head that writes B is zero-initialised, so B A = 0 and the adapted model IS the
# base model at this point. Everything the adapter later does is measured from here.
with torch.no_grad():
    got = tc(probe_x, probe_t, COND_EMB[probe_y], cond_ids=COND(probe_y))
assert torch.allclose(got, ref, atol=1e-5), (got - ref).abs().max().item()
print("no-op at init confirmed")

assert all(n.startswith("hyper.") for n, p in tc.named_parameters() if p.requires_grad)
N_FROZEN = sum(p.numel() for p in base_frozen.parameters() if not p.requires_grad)
print(f"trainable: {tc.n_trainable():,} params, all in the hypernetwork")
print(f"frozen:    {N_FROZEN:,} params")
print(f"adapted layers: {len(tc.adapted)}")
print(f"label path: {'ON (true label)' if cfg.label_path else 'OFF (null label; text only)'}")

In [ ]:
def tc_train(x, y):
    # App. B: the ordinary noise-prediction loss. Only the hypernetwork has requires_grad, so
    # B and A are trained indirectly, through the network that writes them. The hypernetwork
    # reads h(c); the frozen backbone gets COND(y).
    xk, k, eps = add_noise(x)
    return F.mse_loss(tc(xk, k.float() / cfg.K, COND_EMB[y], cond_ids=COND(y)), eps)


def tc_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(tc(xk, k.float() / cfg.K, COND_EMB[y], cond_ids=COND(y)), eps)


tc_hist = run_epochs(tc, [p for p in tc.parameters() if p.requires_grad],
                     cfg.lora_epochs, cfg.lr_hyper, "tc-lora", tc_train, tc_val, "tclora.pt")

# restore the best hypernetwork before sampling
best = torch.load("tclora.pt", map_location=DEV)["state"]
tc.hyper.load_state_dict({k[len("hyper."):]: v for k, v in best.items()
                          if k.startswith("hyper.")})
tc.eval()
print("restored best hypernetwork")

## The comparable numbers, and the diagnostic that matters

Three validation MSEs on identical examples, timesteps and noise:

- **base, same label path** — what the adapted model starts from. With the label path off
  this is the *unconditional* model.
- **base, true label** — a reference: what knowing the class is worth when it comes in
  through the label embedding.
- **TC-LoRA** — the arm under test.

And the diagnostic: validation with the **wrong text**. Labels are shifted by a fixed
offset, so every example gets a genuinely different class, and the shift consumes no random
state, so correct- and wrong-text passes see byte-identical noise and timesteps. Averaged
over three offsets. Same cells as in the cross-attention notebook, so the two arms read the
same way.

Read the result as *average loss sensitivity to the text*. Base-vs-adapter differences
include the adaptation itself; correct-vs-wrong text *within* the adapted model is what
isolates the text. A null here is reportable.

In [ ]:
def frozen_val_same_path(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(base_frozen(xk, k.float() / cfg.K, COND(y)), eps)


def frozen_val_label(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(base_frozen(xk, k.float() / cfg.K, y), eps)


def tc_val_wrong_text(offset):
    '''Wrong text via a fixed label shift: always a different class, never touches gen.
    Only the text changes. Noise, timesteps and the backbone's label input are identical to
    the correct-text pass, so any loss difference is the text.'''
    def fn(x, y, gen):
        xk, k, eps = add_noise(x, gen)
        y_wrong = (y + offset) % meta["n_conditions"]
        return F.mse_loss(tc(xk, k.float() / cfg.K, COND_EMB[y_wrong], cond_ids=COND(y)), eps)
    return fn


WRONG_OFFSETS = (1, 37, 73) if meta["n_conditions"] > 73 else tuple(
    range(1, meta["n_conditions"]))

VAL_BASE_SAME = validate(base_frozen, frozen_val_same_path)
VAL_BASE_LABEL = validate(base_frozen, frozen_val_label)
VAL_TC = validate(tc, tc_val)
VAL_TC_WRONG_EACH = [validate(tc, tc_val_wrong_text(o)) for o in WRONG_OFFSETS]
VAL_TC_WRONG = sum(VAL_TC_WRONG_EACH) / len(VAL_TC_WRONG_EACH)

print(f"base, same label path as tc-lora : {VAL_BASE_SAME:.4f}")
print(f"base, true label (reference)     : {VAL_BASE_LABEL:.4f}")
print(f"tc-lora, correct text            : {VAL_TC:.4f}   ({(VAL_BASE_SAME - VAL_TC) / VAL_BASE_SAME:+.1%} vs base)")
print(f"tc-lora, wrong text (mean of {len(WRONG_OFFSETS)}) : {VAL_TC_WRONG:.4f}   "
      f"per offset {[round(v, 4) for v in VAL_TC_WRONG_EACH]}")
TEXT_EFFECT = (VAL_TC_WRONG - VAL_TC) / VAL_TC
print()
print(f"text effect (wrong - correct) / correct = {TEXT_EFFECT:+.2%}")
if abs(TEXT_EFFECT) < 0.005:
    print("little average loss sensitivity to the text at this training budget. "
          "Check the per-noise-level breakdown and the sample comparison below before concluding.")

## Text sensitivity by noise level

The aggregate number above averages over the whole trajectory. Text may matter only late
(fine detail) or only early (layout). This bins the correct-vs-wrong loss gap by noise
level on the same fixed noise.

In [ ]:
@torch.no_grad()
def loss_by_noise_level(offset, n_bins=5):
    tc.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    edges = torch.linspace(0, cfg.K, n_bins + 1, device=DEV)
    sums = torch.zeros(n_bins, 2, device=DEV); counts = torch.zeros(n_bins, device=DEV)
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        xk, k, eps = add_noise(x, gen)
        t = k.float() / cfg.K
        yw = (y + offset) % meta["n_conditions"]
        e_ok = ((tc(xk, t, COND_EMB[y], cond_ids=COND(y)) - eps) ** 2).flatten(1).mean(1)
        e_wr = ((tc(xk, t, COND_EMB[yw], cond_ids=COND(y)) - eps) ** 2).flatten(1).mean(1)
        b = torch.bucketize(k.float(), edges[1:-1])
        sums.index_add_(0, b, torch.stack([e_ok, e_wr], 1)); counts.index_add_(0, b, torch.ones_like(b, dtype=torch.float))
    return edges.cpu(), (sums / counts[:, None]).cpu()

edges, per_bin = loss_by_noise_level(WRONG_OFFSETS[0])
print(f"{'noise level (k)':>18} {'correct':>9} {'wrong':>9} {'gap':>8}")
TEXT_EFFECT_BY_BIN = []
for i in range(len(per_bin)):
    ok, wr = per_bin[i].tolist()
    TEXT_EFFECT_BY_BIN.append((wr - ok) / ok)
    print(f"{int(edges[i]):>7}-{int(edges[i+1]):<9} {ok:>9.4f} {wr:>9.4f} {(wr-ok)/ok:>+8.2%}")

## Did the adapter actually move?

A flat TC-LoRA loss curve has two very different explanations: the hypernetwork never left
its zero initialisation (learning rate too low), or it moved and there was nothing to learn.
Only the second is a result. This cell tells them apart by measuring the generated adapter
`B A` relative to the frozen weight it modifies, per layer.

Ratios near 0.001 or below mean the adapter never moved — raise `lr_hyper` and retrain the
TC-LoRA cells. Ratios of 0.05 and above mean it did move; if the loss still did not improve,
the adapter had nothing to add.

In [ ]:
with torch.no_grad():
    y = cb[:16].to(DEV)
    t = torch.rand(16, device=DEV)
    a, b = tc.hyper(t, COND_EMB[y], tc._idx, tc._kind)
    ratios = []
    print(f"{'layer':>5} {'shape':>12} {'||BA|| / ||W||':>15}")
    for i, layer in enumerate(tc.adapted):
        d_out, d_in = layer.shape
        ba = torch.einsum("bor,bri->boi", b[:, i, :d_out, :], a[:, i, :, :d_in]) * layer.scale
        W = layer.base.weight
        W = W[:, :, W.shape[2] // 2, W.shape[3] // 2] if W.dim() == 4 else W   # conv: centre tap
        r = (ba.norm(dim=(1, 2)) / W.norm()).mean().item()
        ratios.append(r)
        print(f"{i:>5} {str(tuple(layer.shape)):>12} {r:>15.4f}")

ADAPTER_REL_NORM = sum(ratios) / len(ratios)
print()
print(f"mean ||BA|| / ||W|| = {ADAPTER_REL_NORM:.4f}")
if ADAPTER_REL_NORM < 0.005:
    print("adapter barely moved from init. Raise lr_hyper before reading anything into the loss.")

## Training curves

Train and validation loss per epoch, with the selected checkpoint marked. This is how the
reported checkpoint was chosen: lowest validation loss, nothing else.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, (name, hist) in zip(ax, [("base", base_hist), ("tc-lora", tc_hist)]):
    ep = [h["epoch"] for h in hist]
    if len(hist) > 1:                      # a reused base.pt has a single point
        a.plot(ep, [h["train"] for h in hist], label="train")
    a.plot(ep, [h["val"] for h in hist], label="val", marker="o" if len(hist) == 1 else None)
    best = min(hist, key=lambda h: h["val"])
    a.axvline(best["epoch"], ls="--", c="grey", lw=1)
    a.scatter([best["epoch"]], [best["val"]], c="red", zorder=5,
              label=f"selected (epoch {best['epoch']})")
    a.set_title(f"{name}: best val {best['val']:.4f}")
    a.set_xlabel("epoch"); a.set_ylabel("noise-prediction MSE"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Samples — same starting noise, three conditions

Row 1: the base model, same label path as the arm under test. Row 2: TC-LoRA with the
**correct** description. Row 3: TC-LoRA with a **wrong** description (labels shifted by
one). All three start from the same noise, so any difference is the text.

In [ ]:
# Five conditions with the most validation cells, 200 samples each, so every metric
# below is per condition against real cells of that condition.
counts = torch.bincount(val_loader.dataset.cond, minlength=meta["n_conditions"])
counts[meta["cond_ids_table"][:, 0] == 0] = 0                 # drop DMSO conditions
EVAL_CONDS = counts.topk(min(5, int((counts > 0).sum()))).indices.tolist()
PER = 200
y_eval = torch.tensor(EVAL_CONDS, device=DEV).repeat_interleave(PER)
y_wrong = y_eval.roll(PER)                        # each block gets the next condition's text
N = len(y_eval)
print("conditions:", [meta["cond_text"][c].split(",")[0] for c in EVAL_CONDS])
g = torch.Generator(device=DEV).manual_seed(cfg.seed + 1)
X0 = torch.randn(N, *meta["shape"], device=DEV, generator=g)

base_s = ddim_sample(lambda x, t: base_frozen(x, t, COND(y_eval)), N, meta["shape"],
                     cfg.sample_steps, x_init=X0)
tc_s = ddim_sample(lambda x, t: tc(x, t, COND_EMB[y_eval], cond_ids=COND(y_eval)),
                   N, meta["shape"], cfg.sample_steps, x_init=X0)
tc_wrong = ddim_sample(lambda x, t: tc(x, t, COND_EMB[y_wrong], cond_ids=COND(y_eval)),
                       N, meta["shape"], cfg.sample_steps, x_init=X0)
print("sampled:", tuple(base_s.shape), "| NFE per sample:", cfg.sample_steps,
      "| paired: same X0 for all rows")

SAMPLE_TEXT_DIFF = ((tc_s - tc_wrong).flatten(1).norm(dim=1) / tc_s.flatten(1).norm(dim=1)).mean().item()
print(f"mean relative difference between correct-text and wrong-text samples: {SAMPLE_TEXT_DIFF:.1%}")

In [ ]:
import matplotlib.pyplot as plt

real_x = val_loader.dataset.x.numpy(); real_c = val_loader.dataset.cond.numpy()
gen_c = y_eval.cpu().numpy()
cmap = plt.get_cmap("tab10")
fig, ax = plt.subplots(1, 3, figsize=(15, 4.4), sharex=True, sharey=True)
for a, (name, pts, cs) in zip(ax, [("real (val)", real_x, real_c), ("base", base_s.cpu().numpy(), gen_c),
                                   ("tc-lora", tc_s.cpu().numpy(), gen_c)]):
    for j, c in enumerate(EVAL_CONDS):
        p = pts[cs == c][:PER]
        a.scatter(p[:, 0], p[:, 1], s=6, alpha=0.5, color=cmap(j),
                  label=meta["cond_text"][c].split(",")[0] if a is ax[0] else None)
    a.set_title(name); a.set_xlabel("PC1")
ax[0].set_ylabel("PC2"); ax[0].legend(fontsize=7, markerscale=2); plt.tight_layout(); plt.show()

## Manifest

In [ ]:
def mmd_rbf(x, y, bandwidth=None, max_n=500, seed=0):
    rng = np.random.default_rng(seed)
    if len(x) > max_n: x = x[rng.choice(len(x), max_n, replace=False)]
    if len(y) > max_n: y = y[rng.choice(len(y), max_n, replace=False)]
    z = np.vstack([x, y])
    d2 = ((z[:, None] - z[None]) ** 2).sum(-1)
    if bandwidth is None:
        med = np.median(d2[d2 > 0])
        bandwidth = float(np.sqrt(med / 2)) if med > 0 else 1.0
    k = np.exp(-d2 / (2 * bandwidth ** 2))
    n, m = len(x), len(y)
    kxx = (k[:n, :n].sum() - np.trace(k[:n, :n])) / max(n * (n - 1), 1)
    kyy = (k[n:, n:].sum() - np.trace(k[n:, n:])) / max(m * (m - 1), 1)
    return float(kxx + kyy - 2 * k[:n, n:].mean()), bandwidth


def energy_distance(x, y, max_n=500, seed=0):
    '''2 E|x-y| - E|x-x'| - E|y-y'|; zero iff the distributions agree.'''
    rng = np.random.default_rng(seed)
    if len(x) > max_n: x = x[rng.choice(len(x), max_n, replace=False)]
    if len(y) > max_n: y = y[rng.choice(len(y), max_n, replace=False)]
    d = lambda a, b: np.sqrt(((a[:, None] - b[None]) ** 2).sum(-1)).mean()
    return float(2 * d(x, y) - d(x, x) - d(y, y))


def variance_ratio(pred, real):
    '''Total variance of the samples over that of the real cells; far below 1 is collapse.'''
    return float(np.trace(np.cov(pred, rowvar=False)) / np.trace(np.cov(real, rowvar=False)))


report = {
    "method": "TC-LoRA (reimplementation of arXiv:2510.09561)",
    "modality": "cells",
    "code_version": GIT_SHA,
    "config": asdict(cfg),
    "label_path": cfg.label_path,
    "val_matched_base_same_path": VAL_BASE_SAME,     # fixed noise, same examples
    "val_matched_base_true_label": VAL_BASE_LABEL,
    "val_matched_tclora": VAL_TC,
    "val_matched_tclora_wrong_text": VAL_TC_WRONG,
    "val_matched_tclora_wrong_text_per_offset": dict(zip(map(str, WRONG_OFFSETS), VAL_TC_WRONG_EACH)),
    "text_effect_rel": TEXT_EFFECT,
    "text_effect_rel_by_noise_bin": TEXT_EFFECT_BY_BIN,
    "sample_text_diff_rel": SAMPLE_TEXT_DIFF,
    "base_best_val_during_training": min(h["val"] for h in base_hist),
    "tclora_best_val_during_training": min(h["val"] for h in tc_hist),
    "trainable_hypernetwork": tc.n_trainable(),
    "frozen_backbone": N_FROZEN,
    "checkpoint_meta": CKPT_META,
    "adapted_layers": len(tc.adapted),
    "adapter_rel_norm": ADAPTER_REL_NORM,
    "nfe": cfg.sample_steps,
    "clip_fallback": IS_FALLBACK,
    "train_seconds": {"base": sum(h.get("seconds", 0) for h in base_hist),
                      "tclora": sum(h.get("seconds", 0) for h in tc_hist)},
    "hardware": torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu",
    "device": str(DEV),
}
report.update(globals().get("FID_SCORES", {}))

# Per condition, against the real validation cells of that condition. Quality (MMD, energy
# distance) and diversity (variance ratio) side by side, as the evaluation protocol asks. The
# wrong-text row is scored against the condition that was REQUESTED (y_eval): if the text is
# read, it should be worse than the correct-text row.
real_x = val_loader.dataset.x.numpy(); real_c = val_loader.dataset.cond.numpy()
gen_c = y_eval.cpu().numpy()
print(f"{'condition':32s} {'arm':18s} {'MMD':>7} {'energy':>7} {'var ratio':>9} {'n real':>7}")
for name, s in [("base", base_s), ("tclora", tc_s), ("tclora_wrong_text", tc_wrong)]:
    arr = s.cpu().numpy(); rows = []
    for c in EVAL_CONDS:
        r = real_x[real_c == c]; gsel = arr[gen_c == c]
        m, bw = mmd_rbf(gsel, r)
        rows.append({"cond": int(c), "text": meta["cond_text"][c], "mmd": m, "mmd_bandwidth": bw,
                     "energy": energy_distance(gsel, r), "var_ratio": variance_ratio(gsel, r), "n_real": int(len(r))})
        print(f"{meta['cond_text'][c][:32]:32s} {name:18s} {m:7.3f} {rows[-1]['energy']:7.3f} {rows[-1]['var_ratio']:9.2f} {len(r):7d}")
    report[f"{name}_per_condition"] = rows
    for k in ("mmd", "energy", "var_ratio"):
        report[f"{name}_{k}_mean"] = float(np.mean([r[k] for r in rows]))
report["eval_conditions"] = EVAL_CONDS
report["samples_per_condition"] = PER

print(json.dumps(report, indent=2))
with open("manifest_tclora_cells.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- [ ] `condition` and `dose` stated next to every number; validation cells are unseen (cell line, drug) pairs, say so
- [ ] MMD and energy distance reported with the variance ratio, never alone; sample counts per condition stated
- [ ] `clip_fallback: false` in the manifest
- [ ] `base_epochs` and `lora_epochs` at their real values, `limit` at 0
- [ ] `label_path` stated next to every number; the two settings are different experiments
- [ ] `text_effect_rel` reported with its sign and the per-noise-level breakdown; a small or null effect is a reportable result, not a reason to withhold the row
- [ ] Three seeds (`seed`), mean +/- std, with `split_seed` held fixed
- [ ] Parameter counts and NFE next to every number
- [ ] Described as a **reimplementation** of arXiv:2510.09561, not a reproduction, with the scale gap disclosed: the authors train 251M hypernetwork parameters for 3 days on 8x H100